In [39]:
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
import json
import pandas as pd
from utils import *

In [40]:
df1 = pd.read_csv("../../../../data/en/encoded/aeda_encoded.csv")
df2 = pd.read_csv("../../../../data/en/encoded/backtranslated_encoded.csv")
df3 = pd.read_csv("../../../../data/en/encoded/translated_encoded.csv")

df = pd.concat([df1, df2, df3], axis=0, ignore_index=True)

In [41]:
df[:5]

,Unnamed: 0,id_EXIST,lang,tweet,number_annotators,annotators,gender_annotators,age_annotators,ethnicities_annotators,study_levels_annotators,countries_annotators,labels_task1_1,labels_task1_2,labels_task1_3,split,source,gold_labels_task1_1,gold_labels_task1_2,gold_labels_task1_3,translated_text
0,0,200001,en,"ffs! how about laying , the blame on the . bas...",6,"['Annotator_391', 'Annotator_392', 'Annotator_...","['F', 'F', 'M', 'M', 'M', 'F']","['18-22', '23-45', '18-22', '23-45', '46+', '4...","['White or Caucasian', 'Black or African Ameri...","['High school degree or equivalent', 'Bachelor...","['Latvia', 'South Africa', 'Poland', 'Mexico',...","['YES', 'YES', 'NO', 'NO', 'YES', 'NO']","['JUDGEMENTAL', 'JUDGEMENTAL', '-', '-', 'REPO...","[['MISOGYNY-NON-SEXUAL-VIOLENCE'], ['SEXUAL-VI...",TRAIN_EN,aeda,NaN,NaN,NaN,NaN
1,1,200002,en,writing . a uni essay ! in ! my local pub with...,6,"['Annotator_391', 'Annotator_392', 'Annotator_...","['F', 'F', 'M', 'M', 'M', 'F']","['18-22', '23-45', '18-22', '23-45', '46+', '4...","['White or Caucasian', 'Black or African Ameri...","['High school degree or equivalent', 'Bachelor...","['Latvia', 'South Africa', 'Poland', 'Mexico',...","['YES', 'YES', 'YES', 'NO', 'YES', 'YES']","['REPORTED', 'DIRECT', 'REPORTED', '-', 'REPOR...","[['STEREOTYPING-DOMINANCE', 'OBJECTIFICATION',...",TRAIN_EN,aeda,1.0,2.0,"[0, 0, 0, 1, 1, 0]",NaN
2,2,200003,en,it is 2021 not 1921. i : dont appreciate that ...,6,"['Annotator_397', 'Annotator_398', 'Annotator_...","['F', 'F', 'M', 'M', 'M', 'F']","['18-22', '23-45', '18-22', '23-45', '46+', '4...","['White or Caucasian', 'White or Caucasian', '...","['High school degree or equivalent', 'Master’s...","['Portugal', 'Portugal', 'Poland', 'Greece', '...","['YES', 'YES', 'NO', 'YES', 'NO', 'YES']","['REPORTED', 'REPORTED', '-', 'REPORTED', '-',...","[['OBJECTIFICATION', 'SEXUAL-VIOLENCE'], ['STE...",TRAIN_EN,aeda,1.0,2.0,"[0, 0, 0, 1, 1, 0]",NaN
3,3,200004,en,this is . unacceptable. use her title as you d...,6,"['Annotator_403', 'Annotator_404', 'Annotator_...","['F', 'F', 'M', 'M', 'M', 'F']","['18-22', '23-45', '18-22', '23-45', '46+', '4...","['White or Caucasian', 'White or Caucasian', '...","['Bachelor’s degree', 'Master’s degree', 'High...","['Portugal', 'United Kingdom', 'Poland', 'Latv...","['YES', 'YES', 'NO', 'YES', 'NO', 'NO']","['REPORTED', 'JUDGEMENTAL', '-', 'JUDGEMENTAL'...","[['MISOGYNY-NON-SEXUAL-VIOLENCE'], ['STEREOTYP...",TRAIN_EN,aeda,NaN,NaN,NaN,NaN
4,4,200005,en,making yourself a harder target . basically bo...,6,"['Annotator_409', 'Annotator_410', 'Annotator_...","['F', 'F', 'M', 'M', 'M', 'F']","['18-22', '23-45', '18-22', '23-45', '46+', '4...","['White or Caucasian', 'White or Caucasian', '...","['Bachelor’s degree', 'Master’s degree', 'High...","['Estonia', 'Romania', 'Slovenia', 'Greece', '...","['YES', 'YES', 'NO', 'NO', 'NO', 'YES']","['JUDGEMENTAL', 'DIRECT', '-', '-', '-', 'DIRE...","[['SEXUAL-VIOLENCE', 'MISOGYNY-NON-SEXUAL-VIOL...",TRAIN_EN,aeda,NaN,NaN,NaN,NaN


In [42]:
df = preprocess_dataframe(df, 'tweet')

In [140]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from collections import Counter
import re
import numpy as np
import gensim.downloader as api
from sklearn.metrics import f1_score, accuracy_score

# --- Preprocessing function ---
def preprocess_tweet(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r"@\w+", '', text)
    text = re.sub(r"#\w+", '', text)
    return text

# --- Dataset Class ---
class SentimentDataset(Dataset):
    def __init__(self, df, vocab, max_length, label_columns):
        self.texts = df['text'].values
        self.labels = df[label_columns].astype(int).values
        self.vocab = vocab
        self.max_length = max_length
        self.pad_token_id = self.vocab.get('<PAD>', 0)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        tokens = self.tokenize_text(text)

        # Convert one-hot row to class index
        label = int(self.labels[idx].argmax())  # argmax gives class index

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

    def tokenize_text(self, text):
        words = re.findall(r'\b\w+\b', text.lower())
        tokens = [self.vocab.get(word, self.vocab.get('<UNK>', 1)) for word in words]

        if len(tokens) < self.max_length:
            tokens.extend([self.pad_token_id] * (self.max_length - len(tokens)))
        else:
            tokens = tokens[:self.max_length]
        return tokens

# --- Vocabulary building with gensim glove-twitter ---
def build_vocab(texts, word2vec_model=None, min_freq=2, embedding_dim=200):
    all_words = []
    for text in texts:
        words = re.findall(r'\b\w+\b', str(text).lower())
        all_words.extend(words)

    word_counts = Counter(all_words)

    vocab = {'<PAD>': 0, '<UNK>': 1}
    embeddings = [np.zeros(embedding_dim, dtype=np.float32),  # PAD embedding
                  np.random.normal(scale=0.6, size=embedding_dim).astype(np.float32)]  # UNK embedding

    for word, count in word_counts.items():
        if count >= min_freq:
            vocab[word] = len(vocab)
            if word2vec_model and word in word2vec_model.key_to_index:
                embeddings.append(word2vec_model[word].astype(np.float32))
            else:
                embeddings.append(np.random.normal(scale=0.6, size=embedding_dim).astype(np.float32))

    embedding_matrix = torch.tensor(np.array(embeddings), dtype=torch.float32)
    return vocab, embedding_matrix

# --- Training function ---
def train_model(model, train_df, eval_df, device,
                max_length=100, epochs=5, batch_size=32, lr=0.001,
                weight_decay=1e-5, clip_grad_norm=1.0):

    vocab = build_vocab(train_df['text'])[0]
    pad_token_id = vocab.get('<PAD>', 0)

    train_dataset = SentimentDataset(train_df, vocab, max_length, classes_task1_1)
    eval_dataset = SentimentDataset(eval_df, vocab, max_length, classes_task1_1)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    eval_loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False)

    criterion = nn.CrossEntropyLoss(weight=torch.tensor([0.1190, 0.7180, 2.1553, 2.9077]).to(device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    model.to(device)
    print(f"Training on {device}...")
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            logits = model(input_ids)
            loss = criterion(logits, labels)
            loss.backward()

            if clip_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad_norm)

            optimizer.step()
            train_loss += loss.item()

        model.eval()
        eval_loss = 0
        correct = 0
        total = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in eval_loader:
                input_ids = batch['input_ids'].to(device)
                labels = batch['labels'].to(device)

                logits = model(input_ids)
                loss = criterion(logits, labels)
                eval_loss += loss.item()

                predicted = torch.argmax(logits, dim=1)

                total += labels.size(0)
                correct += (predicted == labels).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        # Compute F1 score
        #f1 = f1_score(all_labels, all_preds, average='macro')  # Binary F1
        macro_f1 = f1_score(all_labels, all_preds, average='macro')  # Macro F1
        per_class_f1 = f1_score(all_labels, all_preds, average=None)  # Returns [F1_class_0, F1_class_1]

        acc = accuracy_score(all_labels, all_preds)

        print(f'Epoch {epoch+1}/{epochs}:')
        print(f'  Train Loss: {train_loss/len(train_loader):.4f}')
        print(f'  Eval Loss: {eval_loss/len(eval_loader):.4f}')
        #print(f'  Eval Accuracy: {100*correct/total:.2f}%')
        print(f'  Eval Accuracy: {100*acc:.2f}%')
        #print(f'  Eval F1 Score (binary): {f1:.4f}')
        print(f'  Eval F1 Score (macro):  {macro_f1:.4f}')
        print(f'  Per-Class F1 Scores: class 0 = {per_class_f1[0]:.4f}, class 1 = {per_class_f1[1]:.4f}, class 2 = {per_class_f1[2]:.4f}, class 3 = {per_class_f1[3]:.4f}')
        print("-" * 30)


In [161]:
import nltk
from nltk.tokenize import word_tokenize
import gensim.downloader as api
import sys
import os

# Add the parent folder of 'models' to sys.path
sys.path.append(os.path.abspath('../'))

from pytorch.attention_models import *
from pytorch.vanilla_models import *

def preprocess_tweet(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r"@\w+", '', text)
    text = re.sub(r"#\w+", '', text)
    #text = re.sub(r"[^a-z\s]", '', text)
    #tokens = nltk.word_tokenize(text)
    return text


df_t = df_train.copy()
df_e = df_eval.copy()

df_t['text'] = df_t['text'].apply(preprocess_tweet)
df_e['text'] = df_e['text'].apply(preprocess_tweet)

# Build vocab and embedding matrix using pretrained vectors
vocab, embedding_matrix = build_vocab(df_t['text'], word2vec_model=word2vec, embedding_dim=100)

# Instantiate model with pretrained embeddings, freezing embeddings
model_cnn = TextCNN(vocab_size=len(vocab),
                embedding_dim=embedding_matrix.shape[1],
                num_classes=4,
                pretrained_embeddings=torch.tensor(embedding_matrix, dtype=torch.float),
                freeze_embeddings=False)

model_lstm = BiLSTMClassifier(vocab_size=len(vocab),
                        hidden_dim=128,
                        embedding_dim=embedding_matrix.shape[1],
                        num_classes=4,
                        num_layers=1,
                        pretrained_embeddings=torch.tensor(embedding_matrix, dtype=torch.float),
                        freeze_embeddings=False)

model_gru = BiGRUClassifier(vocab_size=len(vocab),
                        hidden_dim=128,
                        embedding_dim=embedding_matrix.shape[1],
                        num_classes=4,
                        num_layers=1,
                        pretrained_embeddings=torch.tensor(embedding_matrix, dtype=torch.float),
                        freeze_embeddings=False)

model_lstm = BiLSTMAttentionClassifier(vocab_size=len(vocab),
                        hidden_dim=128,
                        embedding_dim=embedding_matrix.shape[1],
                        num_classes=4,
                        pretrained_embeddings=torch.tensor(embedding_matrix, dtype=torch.float),
                        freeze_embeddings=False)

model_gru = BiGRUAttentionClassifier(vocab_size=len(vocab),
                        hidden_dim=128,
                        embedding_dim=embedding_matrix.shape[1],
                        num_classes=4,
                        pretrained_embeddings=torch.tensor(embedding_matrix, dtype=torch.float),
                        freeze_embeddings=False)
                        

# Train model
print("Training TextCNN model")
train_model(model_cnn, df_t, df_e, device=torch.device('cpu'), epochs=5)
print("-" * 30)
print("-" * 30)
print("Training BiLSTM model")
train_model(model_lstm, df_t, df_e, device=torch.device('cpu'), epochs=5)
print("-" * 30)
print("-" * 30)
print("Training BiGRU model")
train_model(model_gru, df_t, df_e, device=torch.device('cpu'), epochs=5)
print("-" * 30)
print("-" * 30)
print("Training BiLSTM model with attention")
train_model(model_lstm, df_t, df_e, device=torch.device('cpu'), epochs=5)
print("-" * 30)
print("-" * 30)
print("Training BiGRU model with attention")
train_model(model_gru, df_t, df_e, device=torch.device('cpu'), epochs=5)

/var/folders/0s/37b847w521n_1v4k0x2wrrlr0000gn/T/ipykernel_39116/542219936.py:36: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pretrained_embeddings=torch.tensor(embedding_matrix, dtype=torch.float),
/var/folders/0s/37b847w521n_1v4k0x2wrrlr0000gn/T/ipykernel_39116/542219936.py:44: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pretrained_embeddings=torch.tensor(embedding_matrix, dtype=torch.float),
/var/folders/0s/37b847w521n_1v4k0x2wrrlr0000gn/T/ipykernel_39116/542219936.py:52: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pretrained_emb

Training TextCNN model
Training on cpu...
Epoch 1/5:
  Train Loss: 1.3506
  Eval Loss: 1.2436
  Eval Accuracy: 26.75%
  Eval F1 Score (macro):  0.2200
  Per-Class F1 Scores: class 0 = 0.2569, class 1 = 0.4343, class 2 = 0.1888, class 3 = 0.0000
------------------------------
Epoch 2/5:
  Train Loss: 1.1114
  Eval Loss: 1.2359
  Eval Accuracy: 62.75%
  Eval F1 Score (macro):  0.4445
  Per-Class F1 Scores: class 0 = 0.7773, class 1 = 0.5634, class 2 = 0.3396, class 3 = 0.0976
------------------------------
Epoch 3/5:
  Train Loss: 0.9263
  Eval Loss: 1.1321
  Eval Accuracy: 55.75%
  Eval F1 Score (macro):  0.4420
  Per-Class F1 Scores: class 0 = 0.7136, class 1 = 0.5283, class 2 = 0.2947, class 3 = 0.2316
------------------------------
Epoch 4/5:
  Train Loss: 0.7830
  Eval Loss: 1.4339
  Eval Accuracy: 69.25%
  Eval F1 Score (macro):  0.4513
  Per-Class F1 Scores: class 0 = 0.8261, class 1 = 0.6293, class 2 = 0.3500, class 3 = 0.0000
------------------------------
Epoch 5/5:
  Train Los